# Stage 3 · WP2 (Part 1): Revising the Stage 1 Knowledge Graph for Graph-Enhanced RAG

The graph-enhanced RAG system of WP2 answers questions from the Stage 1 knowledge graph: an entity in the question,
a typed relation, the sentence that states it, and the article and date. The feedback on Stage 1 named three points
that this path depends on:

| feedback | what this notebook does | section |
|---|---|---|
| keep lexicon-derived knowledge separate from article-supported claims | every edge and assertion gets a knowledge layer; lexicon facts become background facts without a passage; centrality is recomputed on article claims only | 2 |
| validate the relations used for retrieval; a substantial share of the reviewed relations was incorrect | every evidence sentence is checked with a natural language inference (NLI) model: does the sentence state the relation? The check is compared with the human labels of Stage 1 before it is applied | 3 |
| check that retrieved passages support the specific relationships | the retriever keeps and ranks passages by their support score; the Stage 1 retrieval check is repeated and a blind review sheet is exported | 4 |

The result is **knowledge graph v2** (Section 5), the input of the graph-enhanced RAG notebook (WP2 Part 2). The
Stage 1 outputs are not changed: v2 adds columns and files next to them.

| Team member | Sections | Contribution |
|---|---|---|
| Jihyeon Choung | 1 – 6 | knowledge layers, evidence support check, revised retrieval and knowledge graph v2 |

## 1. Setup

Inputs (all from the submitted Stage 1 run):
- `outputs/stage1_knowledge_graph/`: nodes, assertions (one row per evidence sentence), collapsed edges and the
  correction tables of the relation refinement (`correction/`);
- `artifacts/stage1_wp2/`: sentences and entity mentions of the extraction;
- `artifacts/stage1_wp3/review/`: the human review of Stage 1 (random sample of 90 Tier B relations, supplementary
  sample, check of the relations behind the findings).

The NLI scores are cached in `artifacts/stage3_wp2/kg_revision/nli_scores.csv`, so the notebook runs without a GPU
once the cache exists. Scoring the about 7,000 sentence–relation pairs from scratch takes about 15 minutes on an
Apple M5 and a few minutes on a Colab GPU.

In [1]:
import json
import platform
import sys
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import roc_auc_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "stage3_kg_revision").exists())
sys.path.insert(0, str(ROOT / "src"))

from stage3_graph_rag import GraphEvidenceRetriever
from stage3_kg_revision import (NLIVerifier, aggregate_support, build_pairs, edge_layers, label_assertions,
                                lexicon_conflicts, score_pairs, sentence_actor_counts, threshold_table, wilson)
from wp3_knowledge_graph.pipeline import calculate_node_metrics

def first_existing(*candidates):
    return next((p for p in candidates if (p / "outputs" / "stage1_knowledge_graph" / "nodes.parquet").exists()), None)

# A local copy of the submitted Stage 1 run (data/stage1_submitted) or the team repository layout
STAGE1 = first_existing(ROOT / "data" / "stage1_submitted", ROOT)
assert STAGE1 is not None, "Stage 1 outputs not found"
KG_DIR = STAGE1 / "outputs" / "stage1_knowledge_graph"
WP2_DIR = STAGE1 / "artifacts" / "stage1_wp2"
CORE_ARTICLES = next((p for p in [STAGE1 / "open_models_core.parquet", ROOT / "data" / "processed" / "open_models_core.parquet"]
                      if p.exists()), None)
REVIEW_DIR = ROOT / "artifacts" / "stage1_wp3" / "review"
COMPACT_SCHEMA = ROOT / "config" / "wp3_compact_schema.json"
OUTPUT_DIR = ROOT / "outputs" / "stage3_kg_v2"                  # knowledge graph v2 (large files, not in git)
ARTIFACT_DIR = ROOT / "artifacts" / "stage3_wp2" / "kg_revision"  # small tables, NLI cache, review sheet
for directory in [OUTPUT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
NLI_CACHE = ARTIFACT_DIR / "nli_scores.csv"
SUPPORT_THRESHOLD = 0.5   # fixed in advance: entailment is more likely than not; not tuned on the labels
SEED = 42
TRIPLE = ["subject_id", "predicate", "object_id"]

pd.set_option("display.max_colwidth", 140)
print(f"Repository root: {ROOT}\nStage 1 outputs: {STAGE1}")

Repository root: /Users/jihyeon/projects/others/CLT_personal/2026-09-CLT-computational_language
Stage 1 outputs: /Users/jihyeon/projects/others/CLT_personal/2026-09-CLT-computational_language/data/stage1_submitted


In [2]:
nodes = pd.read_parquet(KG_DIR / "nodes.parquet")
assertions = pd.read_parquet(KG_DIR / "typed_edges_assertions.parquet")
collapsed = pd.read_parquet(KG_DIR / "typed_edges_collapsed.parquet")
relations_all_tiers = pd.read_csv(KG_DIR / "correction" / "relations_all_tiers.csv")
evidence_status = pd.read_csv(KG_DIR / "correction" / "evidence_status.csv")
nodes_corrected = pd.read_csv(KG_DIR / "correction" / "nodes_corrected.csv", keep_default_na=False, na_values=[""])
sentences = pd.read_csv(WP2_DIR / "sentences.csv")
mentions = pd.read_csv(WP2_DIR / "entity_mentions.csv", usecols=["entity_id", "doc_id", "sentence_id"])
compact_type_of = json.loads(COMPACT_SCHEMA.read_text(encoding="utf-8"))["entity_type_mapping"]
labels_of = nodes.set_index("entity_id")["canonical_name"]

display(pd.DataFrame([
    ("nodes", len(nodes)), ("assertions (evidence rows)", len(assertions)), ("edges (unique triples)", len(collapsed)),
    ("all extracted triples with tier / exclusion reason", len(relations_all_tiers)),
    ("GLiREL evidence rows", len(evidence_status)), ("sentences", len(sentences)),
], columns=["Stage 1 table", "rows"]))

,Stage 1 table,rows
0,nodes,5092
1,assertions (evidence rows),6053
2,edges (unique triples),999
3,all extracted triples with tier / exclusion reason,22372
4,GLiREL evidence rows,30560
5,sentences,99438


## 2. Knowledge Layers: Article Claims and Lexicon Facts

The Stage 1 graph combines three kinds of knowledge with different meanings:

| layer | source | what it means | passage |
|---|---|---|---|
| `article_claim` | GLiREL relation in an article sentence | an article states the relation | the sentence |
| `name_pattern` | model name with a size (`Qwen3-8B` → member of family `Qwen`) | the name of a model mentioned in an article | the sentence with the name |
| `curated_lexicon` | WP1 lexicon (model family → developer) | knowledge the team curated for matching | none |

Stage 1 stored them in one edge table with quality tiers (A = lexicon, B = articles). Three consequences:
- a triple found both in the lexicon and in articles was kept as **one Tier A row**, so article-supported edges were
  labelled as lexicon edges, and the lexicon row was counted as evidence;
- lexicon edges took part in centrality and communities like article claims;
- the lexicon was also used as a **filter**: an article relation that names a different developer of a lexicon family
  was excluded (`conflicts_lexicon_developer`).

### 2.1 Layer of every edge

In [3]:
assertions = label_assertions(assertions)
edges = edge_layers(collapsed, assertions)
display(pd.crosstab(edges["knowledge_layer"], edges["tier"].rename("Stage 1 tier"), margins=True))

mixed = edges[edges["lexicon_agreement"].eq("lexicon_and_articles")].assign(
    subject=lambda d: d["subject_id"].map(labels_of), object=lambda d: d["object_id"].map(labels_of))
lexicon_only = edges[edges["lexicon_agreement"].eq("lexicon_only")].assign(
    subject=lambda d: d["subject_id"].map(labels_of), object=lambda d: d["object_id"].map(labels_of))
print(f"Lexicon family -> developer edges: {len(mixed) + len(lexicon_only)} "
      f"({len(mixed)} also stated in articles, {len(lexicon_only)} lexicon only)")
display(mixed.sort_values("article_documents", ascending=False)
        [["subject", "predicate", "object", "tier", "evidence_count", "article_sentences", "article_documents"]].head(15))

Stage 1 tier,A,B,All
knowledge_layer,,,
article_claim,15,831,846
curated_lexicon,18,0,18
name_pattern,0,135,135
All,33,966,999


Lexicon family -> developer edges: 33 (15 also stated in articles, 18 lexicon only)


,subject,predicate,object,tier,evidence_count,article_sentences,article_documents
273,Llama,developed by,Meta,A,196,195,114
290,Nemotron,developed by,NVIDIA,A,148,147,70
235,Gemma,developed by,Google,A,86,85,61
309,Phi,developed by,Microsoft,A,31,30,20
268,Kimi,developed by,MoonShot AI,A,24,23,18
311,Qwen,developed by,Alibaba,A,19,18,17
253,gpt-oss,developed by,OpenAI,A,27,26,15
306,OLMo / Molmo / Tülu,developed by,Ai2,A,18,17,13
255,Granite,developed by,IBM,A,36,35,11
259,Hunyuan,developed by,Tencent,A,11,10,6


**Reading.** The rows above were Tier A in Stage 1, although articles state them; `evidence_count` included the
lexicon row. In v2 they are `article_claim` edges with `in_lexicon = True` (`lexicon_agreement =
lexicon_and_articles`), and their support counts only article sentences. The lexicon-only edges keep the
`curated_lexicon` layer: they are background knowledge, not claims of the corpus.

### 2.2 The lexicon as a filter

Stage 1 excluded article relations that give a lexicon model family a different developer. The table counts them and
shows the most frequent; Section 3.5 checks whether their sentences actually state them.

In [4]:
conflicts = lexicon_conflicts(relations_all_tiers)
conflicts["subject"], conflicts["object"] = conflicts["head"], conflicts["tail"]
print(f"Article relations excluded with conflicts_lexicon_developer: {len(conflicts)} "
      f"({int(conflicts['only_reason'].sum())} for this reason only)")
display(conflicts[conflicts["only_reason"]].head(12)[["subject", "relation", "object", "n_distinct_sentences", "n_docs"]])

Article relations excluded with conflicts_lexicon_developer: 435 (249 for this reason only)


,subject,relation,object,n_distinct_sentences,n_docs
3,Llama,developed by,NVIDIA,12.0,11.0
7,gpt-oss,developed by,NVIDIA,7.0,5.0
8,Llama,developed by,DeepSeek,7.0,8.0
10,Qwen,developed by,OpenAI,5.0,5.0
13,Llama,developed by,Microsoft,5.0,5.0
14,Qwen,developed by,Microsoft,4.0,4.0
15,Llama,developed by,Google,4.0,4.0
16,Qwen,developed by,DeepSeek,4.0,3.0
17,Mistral,developed by,NVIDIA,3.0,3.0
20,Gemma,developed by,NVIDIA,3.0,2.0


### 2.3 Centrality with and without lexicon edges

The Stage 1 analysis view (without model-name and attribute edges) is compared with the same view restricted to
article claims, with the number of supporting **articles** as edge weight (lexicon rows no longer count).

In [5]:
compact_of_entity = nodes.set_index("entity_id")["entity_type_compact"]

def analysis_graph(edge_table, weight_column):
    graph = nx.MultiDiGraph()
    for row in edge_table.itertuples(index=False):
        for entity_id in (row.subject_id, row.object_id):
            graph.add_node(entity_id, label=labels_of.get(entity_id, entity_id), entity_type=compact_of_entity.get(entity_id))
        graph.add_edge(row.subject_id, row.object_id, key=row.predicate,
                       weight=max(float(getattr(row, weight_column)), 1.0))
    return graph

analysis_edges = edges[~edges["predicate_group"].eq("ATTRIBUTE") & ~edges["added_by"].eq("model_name_member")]
centrality = {}
for view, table, weight in [("Stage 1 (with lexicon edges)", analysis_edges, "document_count"),
                            ("article claims only", analysis_edges[analysis_edges["has_article_claim"]], "article_documents")]:
    metrics = calculate_node_metrics(analysis_graph(table, weight), view, "weight", sample_size=1000, seed=SEED)
    centrality[view] = metrics.assign(rank=np.arange(1, len(metrics) + 1)).set_index("entity_id")

stage1_view, article_view = centrality
rank_compare = (centrality[stage1_view][["canonical_name", "rank", "pagerank"]].head(20)
                .join(centrality[article_view][["rank", "pagerank"]].add_suffix("_articles")))
rank_compare["rank_change"] = rank_compare["rank"] - rank_compare["rank_articles"]
display(rank_compare.round(4))

,canonical_name,rank,pagerank,rank_articles,pagerank_articles,rank_change
entity_id,,,,,,
org:nvidia,NVIDIA,1,0.0609,1,0.0630,0
org:openai,OpenAI,2,0.0555,2,0.0571,0
org:google,Google,3,0.0476,3,0.0490,0
org:deepseek,DeepSeek,4,0.0386,4,0.0395,0
org:meta,Meta,5,0.0285,5,0.0293,0
org:anthropic,Anthropic,6,0.0245,6,0.0253,0
org:microsoft,Microsoft,7,0.0205,7,0.0212,0
model:llama,Llama,8,0.0106,8,0.0108,0
org:amazon,Amazon,9,0.0101,9,0.0104,0


In [6]:
moved = rank_compare[rank_compare["rank_change"].abs() >= 3]
dropped = rank_compare[rank_compare["rank_articles"].isna()]
display(Markdown(f'''
**Reading.** {len(rank_compare) - len(dropped)} of the 20 highest-PageRank entities of Stage 1 keep a rank in the
article-only view; {len(moved)} move by three or more places{" (" + ", ".join(moved["canonical_name"]) + ")" if len(moved) else ""}.
The lexicon edges are few ({int(edges["in_lexicon"].sum())} of {len(edges)}), so they do not drive the central actors, but rankings that are meant to
describe the articles are computed without them (Section 3.6 also removes `competes with`).
'''))


**Reading.** 20 of the 20 highest-PageRank entities of Stage 1 keep a rank in the
article-only view; 0 move by three or more places.
The lexicon edges are few (33 of 999), so they do not drive the central actors, but rankings that are meant to
describe the articles are computed without them (Section 3.6 also removes `competes with`).


## 3. Does the Sentence State the Relation? Evidence Support Check

### 3.1 Method

The human review of Stage 1 found that about 40 % of the sampled Tier B relations were wrong, mostly because a
relation was attached to the wrong entity in a sentence naming several models or companies. A refinement rule per
error type cannot catch all of these. A sentence-level check can: for every evidence sentence, a natural language
inference (NLI) model decides whether the sentence (premise) entails the relation written as a sentence
(hypothesis).

| relation | hypothesis |
|---|---|
| `Qwen -> developed by -> Alibaba` | "Qwen was developed by Alibaba." |
| `DeepSeek -- competes with -- OpenAI` | "DeepSeek competes with OpenAI." and "OpenAI competes with DeepSeek." (best of both) |

- **Model**: `DeBERTa-v3-large` trained on MNLI, FEVER, ANLI, LingNLI and WANLI, a standard zero-shot claim
  verification model that runs locally (no API), so every score is reproducible.
- **Names**: the hypothesis uses the **graph node names**, not the words of the sentence. A sentence about
  *Llama Nemotron* (NVIDIA) therefore does not support `Llama -> developed by -> NVIDIA`: entity-linking errors become
  visible to the check. For composite family names (`Command R / Aya`) the part that occurs in the sentence is used.
- **Support score** of a sentence = entailment probability. A sentence supports the relation if the score is at
  least **0.5** (entailment more likely than not), a value fixed before comparing with the labels.
- **Lists**: a sentence that names three or more models or organisations does not support `competes with`
  (Section 3.3 shows why).
- **Edge**: an edge is `supported` if at least one of its sentences supports it.
- **Name patterns** (`Qwen3-8B -> member of family -> Qwen`) are not checked: the relation comes from the model
  name, not from what the sentence says, so an NLI model has nothing to verify (a sentence that only names
  *Qwen3-8B* is neutral about its family). The model-name step was checked in the Stage 1 supplementary review
  (13 of 13 correct); these edges keep `support_status = name_pattern`.

### 3.2 Evidence sentences and scoring

Scored: all evidence sentences of the article edges of the graph, all sentences of the relations a person labelled
(including those removed after the review, so that the check can be compared with the labels), and the sentences of
the article relations excluded by the lexicon filter.

In [7]:
def triple_key(subject_id, predicate, object_id):
    # competes with is symmetric: one key for both orders
    if predicate == "competes with":
        subject_id, object_id = sorted([subject_id, object_id])
    return f"{subject_id}|{predicate}|{object_id}"

def with_key(frame):
    return frame.assign(key=[triple_key(*t) for t in frame[TRIPLE].itertuples(index=False, name=None)])

# Human labels of Stage 1
random_sample = pd.read_excel(REVIEW_DIR / "relation_review_corrected_seed2026.xlsx", sheet_name="Review")
supplement = pd.read_excel(REVIEW_DIR / "relation_review_supplement_seed2026.xlsx", sheet_name="Review")
supplement = supplement[supplement["added_by"].eq("role_typed_tier_B")]   # the only supplementary group in the graph
findings_check = pd.read_csv(REVIEW_DIR / "insight_relation_check.csv").rename(columns={"relation": "predicate"})
review_columns = {"head_id": "subject_id", "tail_id": "object_id", "review_error_type": "error_type"}
labels = pd.concat([
    random_sample.rename(columns=review_columns).assign(label=random_sample["review_correct"], label_set="random sample",
                                                        label_source="human"),
    supplement.rename(columns=review_columns).assign(label=supplement["review_correct"], label_set="random sample",
                                                     label_source="human"),
    findings_check.rename(columns=review_columns).assign(
        label=findings_check["verdict"], label_set="findings check",
        label_source=np.where(findings_check["human_checked"], "human", "AI first pass")),
], ignore_index=True)
labels["label"] = labels["label"].astype(str).str.strip().str.upper()
labels = with_key(labels)[["key", *TRIPLE, "label", "label_set", "label_source", "error_type", "text_unit_id"]]
print(pd.crosstab([labels["label_set"], labels["label_source"]], labels["label"]))

label                         AMBIGUOUS  FALSE  TRUE
label_set      label_source                         
findings check AI first pass          1      0   258
               human                  0     60     8
random sample  human                  2     36    55


In [8]:
# Evidence sentences (GLiREL) with the corrected node IDs of Stage 1
glirel = (evidence_status[evidence_status["evidence_status"].eq("ok")]
          .rename(columns={"head_id": "subject_id", "relation": "predicate", "tail_id": "object_id",
                           "head": "subject", "tail": "object", "score": "glirel_score"}))
glirel = with_key(glirel).assign(knowledge_source="article_claim",
                                 text_unit_id=lambda d: d["doc_id"] + ":" + d["sentence_id"].astype(str))
graph_keys = set(with_key(edges[edges["has_article_claim"]])["key"])
conflict_keys = set(with_key(conflicts.rename(columns={"head_id": "subject_id", "relation": "predicate",
                                                       "tail_id": "object_id"}))["key"])
glirel = glirel[glirel["key"].isin(graph_keys | set(labels["key"]) | conflict_keys)]

columns = ["key", *TRIPLE, "subject", "object", "doc_id", "sentence_id", "text_unit_id", "sentence_text", "glirel_score"]
pool = glirel[columns].reset_index(drop=True)

# Named models / organisations per sentence (lists)
id_map = (nodes_corrected.assign(original_id=nodes_corrected["merged_ids"].str.split(" | ", regex=False))
          .explode("original_id").set_index("original_id")["entity_id"])
compact_of_node = nodes_corrected.set_index("entity_id")["entity_type"].map(compact_type_of)
generic_ids = set(nodes_corrected.loc[nodes_corrected["is_generic"].astype(str).eq("True"), "entity_id"])
actors = sentence_actor_counts(mentions, id_map, compact_of_node, generic_ids)
pool["n_actors"] = [int(actors.get((d, s), 0)) for d, s in zip(pool["doc_id"], pool["sentence_id"])]

pairs = build_pairs(pool, head_name="subject", tail_name="object")
print(f"Evidence sentences: {len(pool):,} of {pool['key'].nunique():,} triples -> {len(pairs):,} premise/hypothesis pairs")
scored = score_pairs(pairs, NLIVerifier(NLI_MODEL, batch_size=32), NLI_CACHE, NLI_MODEL)
sentence_support, edge_support = aggregate_support(pool, scored, ["key"], threshold=SUPPORT_THRESHOLD)
print(f"Sentences supporting their relation: {sentence_support['supported'].mean():.1%}; "
      f"list sentences: {sentence_support['list_sentence'].mean():.1%}")

Evidence sentences: 5,726 of 1,380 triples -> 7,062 premise/hypothesis pairs
Sentences supporting their relation: 60.1%; list sentences: 46.8%


### 3.3 Agreement with the human labels

Before the check is applied, it is compared with the Stage 1 labels:
- **random sample** (human): 90 Tier B relations drawn at random before the review, plus the 3 relations of the
  supplementary sample that are in the graph. This is the main evaluation;
- **findings check**: the relations behind the Stage 1 findings. `FALSE` labels were confirmed by a person, `TRUE`
  labels mostly come from an AI first pass, so this set is reported separately.

The baseline is the extraction confidence of GLiREL (mean score of the edge). The ranking quality (AUC) does not
depend on a threshold; precision and recall are given for the fixed threshold 0.5.

In [9]:
glirel_mean = pool.groupby("key")["glirel_score"].mean()
decided = labels[labels["label"].isin(["TRUE", "FALSE"])].merge(edge_support, on="key", how="left", validate="many_to_one")
decided["is_true"] = decided["label"].eq("TRUE")
decided["kept"] = decided["support_status"].eq("supported")
decided["is_competes"] = decided["predicate"].eq("competes with")

def agreement_row(name, group):
    y, kept = group["is_true"], group["kept"]
    tp = int((y & kept).sum())
    low, high = wilson(tp, int(kept.sum()))
    low0, high0 = wilson(int(y.sum()), len(y))
    return {"labels": name, "relations": len(group), "precision without check": f"{y.mean():.2f} ({low0:.2f}-{high0:.2f})",
            "AUC NLI support": round(roc_auc_score(y, group["support_max"]), 3) if y.nunique() == 2 else np.nan,
            "AUC GLiREL score": round(roc_auc_score(y, group["key"].map(glirel_mean)), 3) if y.nunique() == 2 else np.nan,
            "kept": int(kept.sum()),
            "precision of kept": f"{tp / kept.sum():.2f} ({low:.2f}-{high:.2f})" if kept.sum() else "none kept",
            "recall of TRUE": round(tp / y.sum(), 3), "FALSE removed": f"{int((~y & ~kept).sum())} / {int((~y).sum())}"}

agreement = pd.DataFrame([agreement_row(f"{s} · {'competes with' if c else 'other relations'}", g)
                          for (s, c), g in decided.groupby(["label_set", "is_competes"])]
                         + [agreement_row(f"{s} · all", g) for s, g in decided.groupby("label_set")])
display(agreement)

,labels,relations,precision without check,AUC NLI support,AUC GLiREL score,kept,precision of kept,recall of TRUE,FALSE removed
0,findings check · other relations,305,0.81 (0.77-0.85),0.897,0.647,276,0.89 (0.85-0.92),0.992,27 / 57
1,findings check · competes with,21,0.86 (0.65-0.95),0.815,0.556,17,0.94 (0.73-0.99),0.889,2 / 3
2,random sample · other relations,69,0.71 (0.59-0.80),0.916,0.690,62,0.79 (0.67-0.87),1.000,7 / 20
3,random sample · competes with,22,0.27 (0.13-0.48),0.792,0.563,6,0.50 (0.19-0.81),0.500,13 / 16
4,findings check · all,326,0.82 (0.77-0.85),0.890,0.642,293,0.89 (0.85-0.92),0.985,29 / 60
5,random sample · all,91,0.60 (0.50-0.70),0.880,0.676,68,0.76 (0.65-0.85),0.945,20 / 36


In [10]:
# Why lists are excluded for competes with: the same check without the list rule
no_list_rule = sentence_support.assign(supported=sentence_support["support_score"] >= SUPPORT_THRESHOLD)
kept_without_rule = no_list_rule.groupby("key")["supported"].any()
competes = decided[decided["is_competes"]].assign(kept_without_list_rule=lambda d: d["key"].map(kept_without_rule))
display(pd.DataFrame([
    {"labels": s, "relations": len(g), "TRUE": int(g["is_true"].sum()),
     "kept by NLI alone": int(g["kept_without_list_rule"].sum()),
     "TRUE among them": int((g["kept_without_list_rule"] & g["is_true"]).sum()),
     "kept with list rule": int(g["kept"].sum()), "TRUE among them ": int((g["kept"] & g["is_true"]).sum())}
    for s, g in competes.groupby("label_set")]))
display(sentence_support[sentence_support["predicate"].eq("competes with") & sentence_support["list_sentence"]
                         & (sentence_support["support_score"] >= 0.9)]
        .sample(4, random_state=SEED)[["subject", "object", "n_actors", "support_score", "sentence_text"]])

,labels,relations,TRUE,kept by NLI alone,TRUE among them,kept with list rule,TRUE among them
0,findings check,21,18,20,18,17,16
1,random sample,22,6,21,6,6,3


,subject,object,n_actors,support_score,sentence_text
1174,Google,Meta,4,0.993765,"Meta is debuting its first major artificial intelligence model since the costly hiring of Scale AI's Alexandr Wang nine months ago, as t..."
4168,GLM,Kimi,4,0.986685,DeepSeek commentary was mixed: fundraising talks reportedly target a $ 45B valuation led by a major Chinese state-backed semiconductor f...
663,DeepSeek,OpenAI,6,0.987288,"Its launch earlier this year also came as a response to a rapidly expanding field of open competitors, including DeepSeek, Mistral, and ..."
1843,Baidu,DeepSeek,4,0.913730,"""Baidu has always been very supportive of its proprietary business model and was vocal against open-source, but disruptors like DeepSeek..."


**Why the list rule.** For `competes with`, the NLI model accepts almost every sentence that names the two entities
together, including lists such as *"open competitors, including DeepSeek, Mistral, and Meta's Llama"*: it reads
co-listing as rivalry. The reviewers did not, and labelled such relations `FALSE` (mostly `entity_linking`). This is
the error the feedback describes (sentences mentioning several competing models). A sentence naming three or more
models or organisations is therefore not accepted as evidence of a rivalry between two of them; three is the
smallest list and was not tuned (four is shown below as a sensitivity check). Other relations are not affected: a
list sentence can still say *"Qwen, developed by Alibaba, ..."*.

The rule is conservative: models and organisations are counted together, so *"DeepSeek said ... its R1 model competes
with OpenAI's o1"* (four names) is also rejected. Looser definitions keep such sentences but also more of the
reviewers' `FALSE` relations, and with 22 sampled `competes with` relations (6 `TRUE`) the difference cannot be
estimated. `competes with` is therefore not used for retrieval (Section 3.6).

In [11]:
rows = []
for set_name, group in decided.groupby("label_set"):
    table = threshold_table(group["is_true"], group["support_max"], thresholds=(0.1, 0.3, 0.5, 0.7, 0.9))
    rows.append(table.assign(labels=set_name, rule="NLI max score, no list rule"))
display(pd.concat(rows)[["labels", "threshold", "kept", "of", "precision", "precision_ci95", "recall_of_true",
                         "removed_false", "removed_true"]].round(3))

sensitivity = []
for minimum in (3, 4):
    rule = (sentence_support["support_score"] >= SUPPORT_THRESHOLD) & ~(
        sentence_support["predicate"].eq("competes with") & (sentence_support["n_actors"] >= minimum))
    kept_keys = set(sentence_support.loc[rule, "key"])
    for set_name, group in decided[decided["is_competes"]].groupby("label_set"):
        kept = group["key"].isin(kept_keys)
        sensitivity.append({"list = names >=": minimum, "labels": set_name, "kept": int(kept.sum()), "of": len(group),
                            "TRUE kept": int((kept & group["is_true"]).sum()), "TRUE": int(group["is_true"].sum())})
display(pd.DataFrame(sensitivity))

,labels,threshold,kept,of,precision,precision_ci95,recall_of_true,removed_false,removed_true
0,findings check,0.1,301,326,0.884,0.84-0.92,1.000,25,0
1,findings check,0.3,297,326,0.892,0.85-0.92,0.996,28,1
2,findings check,0.5,296,326,0.892,0.85-0.92,0.992,28,2
3,findings check,0.7,295,326,0.895,0.85-0.92,0.992,29,2
4,findings check,0.9,289,326,0.907,0.87-0.93,0.985,33,4
0,random sample,0.1,85,91,0.647,0.54-0.74,1.000,6,0
1,random sample,0.3,84,91,0.655,0.55-0.75,1.000,7,0
2,random sample,0.5,83,91,0.663,0.56-0.76,1.000,8,0
3,random sample,0.7,83,91,0.663,0.56-0.76,1.000,8,0
4,random sample,0.9,78,91,0.692,0.58-0.78,0.982,12,1


,list = names >=,labels,kept,of,TRUE kept,TRUE
0,3,findings check,17,21,16,18
1,3,random sample,6,22,3,6
2,4,findings check,20,21,18,18
3,4,random sample,9,22,4,6


**Reading.** The threshold hardly matters: most scores are close to 0 or 1, so 0.5 is kept. The NLI score ranks
correct above incorrect relations much better than the GLiREL confidence (AUC in the table above), which Stage 1 had
therefore not used as a filter.

In [12]:
by_predicate = (decided.groupby("predicate")
                .agg(labelled=("is_true", "size"), true=("is_true", "sum"), kept=("kept", "sum"),
                     true_kept=("is_true", lambda s: int((s & decided.loc[s.index, "kept"]).sum())))
                .assign(precision_before=lambda d: (d["true"] / d["labelled"]).round(2),
                        precision_after=lambda d: (d["true_kept"] / d["kept"]).round(2),
                        recall=lambda d: (d["true_kept"] / d["true"]).round(2))
                .sort_values("labelled", ascending=False))
display(by_predicate)

false_labels = decided[~decided["is_true"]]
display(pd.crosstab(false_labels["error_type"].fillna("unspecified"), false_labels["kept"].map({True: "kept", False: "removed"}),
                    margins=True).rename_axis(index="error type of FALSE labels", columns="check"))

,labelled,true,kept,true_kept,precision_before,precision_after,recall
predicate,,,,,,,
developed by,274,230,254,228,0.84,0.90,0.99
competes with,43,24,23,19,0.56,0.83,0.79
outperforms,22,14,18,14,0.64,0.78,1.00
works for,19,19,19,19,1.00,1.00,1.00
headquartered in,12,7,9,7,0.58,0.78,1.00
founded by,10,6,8,6,0.60,0.75,1.00
available on,7,6,6,6,0.86,1.00,1.00
supported by,6,1,5,1,0.17,0.20,1.00
acquired by,6,3,4,3,0.50,0.75,1.00


check,kept,removed,All
error type of FALSE labels,,,
duplicate_or_noise,1,0,1
entity_boundary,8,3,11
entity_linking,25,30,55
insufficient_context,3,4,7
negation_or_uncertainty,1,0,1
other,1,2,3
reversed_direction,1,6,7
schema_or_catalogue,1,2,3
wrong_entity,1,0,1


### 3.4 Where the check and the reviewers disagree

In [13]:
def best_sentence(key):
    rows = sentence_support[sentence_support["key"] == key].sort_values("support_score", ascending=False)
    return rows.iloc[0]

disagreements = []
for kind, mask in [("labelled FALSE, check: supported", ~decided["is_true"] & decided["kept"]),
                   ("labelled TRUE, check: not supported", decided["is_true"] & ~decided["kept"])]:
    for row in decided[mask & decided["label_source"].eq("human")].itertuples():
        sentence = best_sentence(row.key)
        disagreements.append({"case": kind, "relation": f"{labels_of.get(row.subject_id)} -> {row.predicate} -> "
                              f"{labels_of.get(row.object_id)}", "error type": row.error_type,
                              "best score": round(float(sentence["support_score"]), 3), "best sentence": sentence["sentence_text"][:230]})
disagreements = pd.DataFrame(disagreements)
print(disagreements["case"].value_counts().to_string())
with pd.option_context("display.max_colwidth", 230):
    display(disagreements.groupby("case").head(8))

case
labelled FALSE, check: supported       47
labelled TRUE, check: not supported     4


,case,relation,error type,best score,best sentence
0,"labelled FALSE, check: supported",ElevenLabs -> competes with -> Google,entity_linking,0.993,"A two-person startup by the name of Nari Labs has introduced Dia, a 1.6 billion parameter text-to-speech (TTS) model designed to produce naturalistic dialogue directly from text prompts — and one of its creators claims it surp..."
1,"labelled FALSE, check: supported",Chrome DevTools -> supported by -> Bun,reversed_direction,0.987,"Bun’s frontend dev server gets “ automatic workspaces ” support in Chrome DevTools, so you can edit files in the browser."
2,"labelled FALSE, check: supported",GKE -> developed by -> NVIDIA,duplicate_or_noise,0.706,"GenAI NVIDIA NeMo\n\nImprove scalability, cost-efficiency, fault tolerance, isolation, and portability by using GKE for Ray workloads."
3,"labelled FALSE, check: supported",GPT-4o -> outperforms -> Claude 3.5 Sonnet,entity_linking,0.782,"Results: UI-TARS matched or outperformed Claude 3.5 Sonnet with computer use, GPT-4o with various computer use frameworks, and the Aguvis framework with its native model on 11 benchmarks."
4,"labelled FALSE, check: supported",Broadcom -> headquartered in -> united_states,insufficient_context,0.996,"Other stocks related to AI, such as Micron, Arm Holdings and Broadcom in the U.S., as well as ASML and Tokyo Electron in global markets, also fell sharply."
5,"labelled FALSE, check: supported",DeepSeek -> founded by -> High-Flyer,entity_linking,0.997,"DeepSeek was founded less than two years ago by the Chinese hedge fund High Flyer as a research lab dedicated to pursuing Artificial General Intelligence, or AGI."
6,"labelled FALSE, check: supported",TrueFoundry -> competes with -> Helicone,insufficient_context,0.857,"While tools like Portkey and Helicone offer gateway features, TrueFoundry stands out as the premier LiteLLM alternative for high-performance needs."
7,"labelled FALSE, check: supported",Frugal AI Hub -> founded by -> Arjuna Sathiaseelan,entity_linking,0.968,"The most compelling example I’ve encountered is the Saving Voices Project, a nonprofit founded by Arjuna Sathiaseelan, who is also the chief technology officer of the Frugal AI Hub at Cambridge University."
47,"labelled TRUE, check: not supported",Google -> competes with -> xAI,NaN,0.991,"The main open questions are whether Anthropic can convert this capacity into sustained product momentum, whether its managed-agent features are truly differentiated, and whether its safety/governance posture helps or hinders i..."
48,"labelled TRUE, check: not supported",Claude -> competes with -> DeepSeek,NaN,0.995,"DeepSeek’s V4 has been keenly anticipated by users looking to test how it compares to U.S. competitors like OpenAI’s ChatGPT, Anthropic’s Claude and Google’s Gemini."


In [14]:
fp = disagreements[disagreements["case"].str.startswith("labelled FALSE")]
display(Markdown(f'''
**Reading.** The check keeps {len(fp)} relations that a person labelled `FALSE`, mostly
{", ".join(f"`{e}` ({n})" for e, n in fp["error type"].fillna("unspecified").value_counts().head(3).items())}. Three
kinds of case recur in the table:
- **fragment nodes** (`Lightning`, `Ultra`, `Pro model`): the sentence states the relation for the full name
  (*NVIDIA Nemotron 3.5 Lightning*), and the hypothesis with the fragment still reads as entailed;
- **possessive lists**: in *"Claude Opus 4, OpenAI's GPT-4o and o4-mini"* the model attaches the owner to the wrong
  name, the same linking error as the extraction;
- **stricter reviewers**: *"DeepSeek was founded ... by the Chinese hedge fund High Flyer"* or *"Broadcom in the
  U.S."* were labelled `FALSE` for the node or the context, although the sentence states the words of the relation.

Only {int((disagreements["case"] == "labelled TRUE, check: not supported").sum())} relations labelled `TRUE` are not
supported, all `competes with` relations whose only sentences are lists. The check is therefore a filter that removes
about half of the errors at almost no loss of correct relations, not a proof of correctness: the precision of the
kept relations (Section 3.3) is the number to report, and errors of the first two kinds remain in the graph.
'''))


**Reading.** The check keeps 47 relations that a person labelled `FALSE`, mostly
`entity_linking` (25), `entity_boundary` (8), `wrong_predicate` (5). Three
kinds of case recur in the table:
- **fragment nodes** (`Lightning`, `Ultra`, `Pro model`): the sentence states the relation for the full name
  (*NVIDIA Nemotron 3.5 Lightning*), and the hypothesis with the fragment still reads as entailed;
- **possessive lists**: in *"Claude Opus 4, OpenAI's GPT-4o and o4-mini"* the model attaches the owner to the wrong
  name, the same linking error as the extraction;
- **stricter reviewers**: *"DeepSeek was founded ... by the Chinese hedge fund High Flyer"* or *"Broadcom in the
  U.S."* were labelled `FALSE` for the node or the context, although the sentence states the words of the relation.

Only 4 relations labelled `TRUE` are not
supported, all `competes with` relations whose only sentences are lists. The check is therefore a filter that removes
about half of the errors at almost no loss of correct relations, not a proof of correctness: the precision of the
kept relations (Section 3.3) is the number to report, and errors of the first two kinds remain in the graph.


### 3.5 Support status of the graph edges

In [15]:
edges = with_key(edges).merge(edge_support, on="key", how="left", validate="one_to_one")
edges.loc[edges["knowledge_layer"].eq("curated_lexicon"), "support_status"] = "no_passage"
edges.loc[edges["knowledge_layer"].eq("name_pattern"), "support_status"] = "name_pattern"
assert edges["support_status"].notna().all()
human_true = edges["review_status"].isin(["reviewed_true", "checked_true_human"])
edges["support_disagrees_with_review"] = human_true & edges["support_status"].eq("not_supported")
# Retrieval uses article claims and name patterns whose sentences state them; a relation a person confirmed is kept
edges["retrieval_eligible"] = (edges["support_status"].isin(["supported", "name_pattern"])
                               | (edges["knowledge_layer"].eq("article_claim") & human_true))

status = (edges.groupby(["knowledge_layer", "predicate"])
          .agg(edges=("key", "size"), supported=("support_status", lambda s: int(s.eq("supported").sum())),
               eligible=("retrieval_eligible", "sum"))
          .assign(share_supported=lambda d: (d["supported"] / d["edges"]).round(2))
          .sort_values("edges", ascending=False))
display(status)
print(f"Edges eligible for retrieval: {int(edges['retrieval_eligible'].sum())} of "
      f"{int(edges['knowledge_layer'].ne('curated_lexicon').sum())} article / name-pattern edges; "
      f"human-confirmed but not supported by the check: {int(edges['support_disagrees_with_review'].sum())}")

conflict_support = with_key(conflicts.rename(columns={"head_id": "subject_id", "relation": "predicate", "tail_id": "object_id"}))
conflict_support = conflict_support.merge(edge_support, on="key", how="left")
print(f"Article relations excluded by the lexicon filter: {conflict_support['support_status'].eq('supported').mean():.0%} "
      f"have a sentence that states them")
display(conflict_support[conflict_support["support_status"].eq("supported") & conflict_support["only_reason"]]
        .head(8)[["subject", "predicate", "object", "n_distinct_sentences", "support_max", "supported_sentences"]])

edges  supported  eligible  share_supported
knowledge_layer predicate                                                    
article_claim   developed by        332        319       319             0.96
name_pattern    member of family    135          0       135             0.00
article_claim   competes with        92         40        44             0.43
                works for            91         87        87             0.96
                headquartered in     57         47        47             0.82
                founded by           53         42        42             0.79
                outperforms          52         41        41             0.79
                available on         31         30        30             0.97
                invested in          28          9         9             0.32
                manufactured by      23         23        23             1.00
curated_lexicon developed by         18          0         0             0.00
article_claim   acquired by          15         10        10             0.67
                supported by         14         14        14             1.00
                announced by         12          8         8             0.67
                runs on              11         10        10             0.91
                licensed under       10         10        10             1.00
                released by           7          7         7             1.00
                subsidiary of         6          4         4             0.67
                used by               4          2         2             0.50
                integrated into       2          2         2             1.00
                based on              2          2         2             1.00
                has capability        1          1         1             1.00
                distilled from        1          1         1             1.00
                complies with         1          0         0             0.00
                regulated by          1          1         1             1.00

Edges eligible for retrieval: 849 of 981 article / name-pattern edges; human-confirmed but not supported by the check: 4
Article relations excluded by the lexicon filter: 41% have a sentence that states them


,subject,predicate,object,n_distinct_sentences,support_max,supported_sentences
3,Llama,developed by,NVIDIA,12.0,0.986118,7
7,gpt-oss,developed by,NVIDIA,7.0,0.985027,4
8,Llama,developed by,DeepSeek,7.0,0.971960,2
10,Qwen,developed by,OpenAI,5.0,0.996497,3
14,Qwen,developed by,Microsoft,4.0,0.987754,2
16,Qwen,developed by,DeepSeek,4.0,0.981931,2
20,Gemma,developed by,NVIDIA,3.0,0.987746,1
28,Qwen,developed by,DavidAU,3.0,0.994939,2


In [16]:
eligible_unreviewed = edges[edges["retrieval_eligible"] & edges["knowledge_layer"].eq("article_claim")
                            & edges["review_status"].eq("unreviewed")]
other = decided[decided["label_set"].eq("random sample") & ~decided["is_competes"]]
competes_rs = decided[decided["label_set"].eq("random sample") & decided["is_competes"]]
p_other = (other["is_true"] & other["kept"]).sum() / other["kept"].sum()
display(Markdown(f'''
**Reading.** {int(edges["retrieval_eligible"].sum())} edges remain eligible for retrieval.
{int(status.loc["article_claim"]["edges"].sum() - status.loc["article_claim"]["supported"].sum())} article edges have
no sentence that states them and are kept in the graph with `support_status = not_supported`, but retrieval no longer
uses them. `competes with` loses most of its edges, as the review suggested (see Section 3.6). For the
{len(eligible_unreviewed)} eligible article edges that no person checked, the random sample gives the expected
precision: {p_other:.2f} for relations other than `competes with` (TRUE among kept), against
{other["is_true"].mean():.2f} without the check. For `competes with`, only {int(competes_rs["kept"].sum())} of
{len(competes_rs)} sampled relations pass, too few for an estimate.

About {conflict_support["support_status"].eq("supported").mean():.0%} of the relations excluded by the lexicon filter
pass the check. Their sentences show why the filter is still needed: `Llama -> developed by -> NVIDIA` comes from
*Llama Nemotron* and *Llama-3.1-Nemotron-70B* (NVIDIA models built on Llama), `Qwen -> developed by -> OpenAI` from
lists such as *"OpenAI's text-embedding-3-large and Qwen3-Embedding-8B"*, `gpt-oss -> developed by -> NVIDIA` from
sentences about running gpt-oss on NVIDIA hardware. The two checks catch different errors: NLI catches relations
the sentence does not state, the lexicon catches family-level linking errors that NLI accepts. The lexicon filter stays,
recorded as a decision based on curated knowledge (`lexicon_conflicts`), not as article evidence.
'''))


**Reading.** 849 edges remain eligible for retrieval.
136 article edges have
no sentence that states them and are kept in the graph with `support_status = not_supported`, but retrieval no longer
uses them. `competes with` loses most of its edges, as the review suggested (see Section 3.6). For the
388 eligible article edges that no person checked, the random sample gives the expected
precision: 0.79 for relations other than `competes with` (TRUE among kept), against
0.71 without the check. For `competes with`, only 6 of
22 sampled relations pass, too few for an estimate.

About 41% of the relations excluded by the lexicon filter
pass the check. Their sentences show why the filter is still needed: `Llama -> developed by -> NVIDIA` comes from
*Llama Nemotron* and *Llama-3.1-Nemotron-70B* (NVIDIA models built on Llama), `Qwen -> developed by -> OpenAI` from
lists such as *"OpenAI's text-embedding-3-large and Qwen3-Embedding-8B"*, `gpt-oss -> developed by -> NVIDIA` from
sentences about running gpt-oss on NVIDIA hardware. The two checks catch different errors: NLI catches relations
the sentence does not state, the lexicon catches family-level linking errors that NLI accepts. The lexicon filter stays,
recorded as a decision based on curated knowledge (`lexicon_conflicts`), not as article evidence.


### 3.6 `competes with` is not used for retrieval or analysis

`competes with` is the relation the evidence cannot validate:
- in the random sample, 6 of 22 relations were correct before any check;
- the NLI check accepts almost all of them, because news sentences list rivals ("Google, OpenAI, and Anthropic ...
  against open-weight models from China"), and the list rule that removes these also removes clear rivalry sentences
  (Section 3.3);
- of 22 sampled relations, 6 pass, 3 of them correct: no rule tested separates correct from incorrect rivalry claims.

A relation that cannot be validated should not answer questions. `competes with` edges therefore stay in the graph
with `relation_reliability = low` and their sentences, but they are not retrieval-eligible and are left out of the
graph metrics. Questions about competition are answered from retrieved text passages (WP1 text retrieval), which
cite what the article says instead of turning it into an edge. Co-mentions of rivals remain visible in the
co-occurrence graph of Stage 1, which does not claim a relation.

In [17]:
LOW_RELIABILITY = {"competes with": "cannot be validated at sentence level: lists of rivals, 6 of 22 sampled relations correct"}
edges["relation_reliability"] = np.where(edges["predicate"].isin(LOW_RELIABILITY), "low", "standard")
edges["retrieval_eligible"] = edges["retrieval_eligible"] & edges["relation_reliability"].eq("standard")
print(f"competes with edges kept in the graph but not used: {int(edges['predicate'].eq('competes with').sum())}; "
      f"edges eligible for retrieval: {int(edges['retrieval_eligible'].sum())}")

# Graph metrics of the retrieval graph (analysis view: no model-name or attribute edges)
retrieval_view = edges[edges["retrieval_eligible"] & edges["has_article_claim"]
                       & ~edges["predicate_group"].eq("ATTRIBUTE") & ~edges["added_by"].eq("model_name_member")]
retrieval_metrics = calculate_node_metrics(analysis_graph(retrieval_view, "article_documents"), "retrieval graph",
                                           "weight", sample_size=1000, seed=SEED)
retrieval_rank = retrieval_metrics.assign(rank_retrieval_graph=np.arange(1, len(retrieval_metrics) + 1)).set_index("entity_id")
rank_shift = (rank_compare[["canonical_name", "rank_articles"]].head(15)
              .join(retrieval_rank[["rank_retrieval_graph", "pagerank"]]))
display(rank_shift.round(4))

competes with edges kept in the graph but not used: 92; edges eligible for retrieval: 805


,canonical_name,rank_articles,rank_retrieval_graph,pagerank
entity_id,,,,
org:nvidia,NVIDIA,1,1,0.0603
org:openai,OpenAI,2,2,0.0388
org:google,Google,3,3,0.0384
org:deepseek,DeepSeek,4,4,0.0274
org:meta,Meta,5,5,0.0215
org:anthropic,Anthropic,6,6,0.0176
org:microsoft,Microsoft,7,9,0.0137
model:llama,Llama,8,7,0.0145
org:amazon,Amazon,9,19,0.0046


In [18]:
dropped = rank_shift[rank_shift["rank_retrieval_graph"] - rank_shift["rank_articles"] >= 5]
competes_degree = edges[edges["predicate"].eq("competes with")].melt(value_vars=["subject_id", "object_id"])["value"].value_counts()
display(Markdown(f'''
**Reading.** Without `competes with`, the leading actors keep their places, but entities whose centrality came
mainly from rivalry edges fall: {", ".join(f"{r.canonical_name} ({int(r.rank_articles)} -> {int(r.rank_retrieval_graph)}, "
f"{int(competes_degree.get(i, 0))} competes with edges)" for i, r in dropped.iterrows())}. Their rank in Stage 1 reflected
how often they were named next to rivals rather than relations the articles state about them. Graph metrics reported
from Stage 3 on use this retrieval graph.
'''))


**Reading.** Without `competes with`, the leading actors keep their places, but entities whose centrality came
mainly from rivalry edges fall: Amazon (9 -> 19, 9 competes with edges), Alibaba (13 -> 29, 10 competes with edges). Their rank in Stage 1 reflected
how often they were named next to rivals rather than relations the articles state about them. Graph metrics reported
from Stage 3 on use this retrieval graph.


## 4. Retrieval Check: Do the Passages Support the Relations?

The retrieval check of Stage 1 (Section 6.9) only tested that a passage is returned. It is repeated with the same
retriever on the Stage 1 assertions (**before**) and on knowledge graph v2 (**after**):
- only retrieval-eligible edges, without lexicon rows; lexicon facts are returned separately as `background`;
- only passages whose sentence supports the relation (support score >= 0.5 and, for `competes with`, not a list),
  ranked by support score.

The questions are the three of Stage 1 plus competency questions of the relation schema.

In [19]:
text_units = pd.DataFrame({"text_unit_id": sentences["doc_id"] + ":" + sentences["sentence_id"].astype(str),
                           "document_id": sentences["doc_id"], "text_unit_text": sentences["sentence_text"]})
documents = (assertions.dropna(subset=["document_id"]).groupby("document_id")["published_at"].first().reset_index())
if CORE_ARTICLES is not None:
    articles = pd.read_parquet(CORE_ARTICLES, columns=["doc_id", "title", "url", "domain"])
    documents = documents.merge(articles.rename(columns={"doc_id": "document_id"}), on="document_id", how="left")

# Knowledge graph v2 assertions: one row per evidence sentence, with layer and support
support_columns = ["support_score", "contradiction_score", "list_sentence", "n_actors", "hypothesis", "supported"]
support_by_unit = sentence_support.drop_duplicates(["key", "text_unit_id"])[["key", "text_unit_id", *support_columns]]
assertions_v2 = (with_key(assertions)
                 .merge(support_by_unit, on=["key", "text_unit_id"], how="left", validate="many_to_one")
                 .merge(edges[["key", "retrieval_eligible"]], on="key", how="left", validate="many_to_one"))
assertions_v2["supported"] = assertions_v2["supported"].fillna(False).astype(bool)
assert assertions_v2.loc[assertions_v2["knowledge_source"].eq("article_claim"), "support_score"].notna().all()
print(assertions_v2.groupby("knowledge_source")[["support_score"]].agg(["count", "mean"]).round(3))

background = (edges[edges["in_lexicon"]][TRIPLE].assign(knowledge_source="curated_lexicon"))
# Passages for retrieval: sentences that support their relation (score and list rule), or name-pattern sentences
retrieval_assertions = assertions_v2[assertions_v2["retrieval_eligible"].fillna(False).astype(bool)
                                     & (assertions_v2["supported"] | assertions_v2["knowledge_source"].eq("name_pattern"))]
print(f"Passages for retrieval: {len(retrieval_assertions):,} of {assertions_v2['text_unit_id'].notna().sum():,} evidence rows")
retriever_before = GraphEvidenceRetriever(nodes, assertions.dropna(subset=["text_unit_id"]), text_units, documents)
retriever_after = GraphEvidenceRetriever(nodes, retrieval_assertions, text_units, documents, background=background)
support_of_relation = assertions_v2.set_index("relation_id")["support_score"]
supported_relation = assertions_v2.set_index("relation_id")["supported"]

                 support_score       
                         count   mean
knowledge_source                     
article_claim             4742  0.764
curated_lexicon              0    NaN
name_pattern                 0    NaN
Passages for retrieval: 4,190 of 6,020 evidence rows


In [20]:
QUESTIONS = [
    ("Who developed Qwen?", ["developed by"]),
    ("Which companies compete with DeepSeek?", ["competes with"]),
    ("What does NVIDIA manufacture?", ["manufactured by"]),
    ("Who developed Gemma?", ["developed by"]),
    ("Who developed Llama?", ["developed by"]),
    ("Which companies compete with OpenAI?", ["competes with"]),
    ("Which models does Kimi outperform?", ["outperforms"]),
    ("Where is DeepSeek headquartered?", ["headquartered in"]),
    ("Who works for Meta?", ["works for"]),
    ("Who founded DeepSeek?", ["founded by"]),
    ("Where is Nemotron available?", ["available on"]),
    ("Who invested in OpenAI?", ["invested in"]),
    ("Under which license is Gemma released?", ["licensed under"]),
]
TOP_K = 3

rows, background_rows = [], []
for question, predicates in QUESTIONS:
    for system, retriever, extra in [("before", retriever_before, {}), ("after", retriever_after, {"min_support": SUPPORT_THRESHOLD})]:
        result = retriever.retrieve(question, predicates=predicates, top_k=TOP_K, **extra)
        for rank, item in enumerate(result["evidence"], start=1):
            rows.append({"question": question, "system": system, "rank": rank, "relation_id": item["relation_id"],
                         "relation": f"{item['subject']['name']} -> {item['predicate']} -> {item['object']['name']}",
                         "support_score": support_of_relation.get(item["relation_id"], np.nan),
                         "published_at": item["published_at"], "document_id": item["document_id"],
                         "passage": " ".join(str(item["passage"]).split())})
        if system == "after":
            background_rows += [{"question": question, "background fact": f"{f['subject']['name']} -> {f['predicate']} -> "
                                 f"{f['object']['name']}", "source": f["knowledge_source"]} for f in result["background"]]
retrieval_check = pd.DataFrame(rows)
retrieval_check["supported"] = retrieval_check["relation_id"].map(supported_relation)
retrieval_check["in_other_system"] = retrieval_check.duplicated(["question", "relation_id"], keep=False)

summary = (retrieval_check.groupby("system")
           .agg(passages=("relation_id", "size"), supported=("supported", "sum"),
                questions_answered=("question", "nunique"), changed=("in_other_system", lambda s: int((~s).sum())))
           .assign(share_supported=lambda d: (d["supported"] / d["passages"]).round(2))
           .reindex(["before", "after"]))
display(summary)
display(pd.DataFrame(background_rows))
with pd.option_context("display.max_colwidth", 170):
    display(retrieval_check[["question", "system", "rank", "relation", "support_score", "supported", "passage"]].round(3))

,passages,supported,questions_answered,changed,share_supported
system,,,,,
before,39,29,13,32,0.74
after,33,33,11,26,1.00


,question,background fact,source
0,Who developed Qwen?,Qwen -> developed by -> Alibaba,curated_lexicon
1,Who developed Gemma?,Gemma -> developed by -> Google,curated_lexicon
2,Who developed Llama?,Llama -> developed by -> Meta,curated_lexicon


,question,system,rank,relation,support_score,supported,passage
0,Who developed Qwen?,before,1,Qwen -> developed by -> Alibaba,0.998,True,"Part of the underlying AI for GeoGPT is Qwen, a large language model built by the Chinese tech company Alibaba."
1,Who developed Qwen?,before,2,Qwen -> developed by -> Alibaba,0.992,True,"Built upon Alibaba’s Qwen3-8B model, this version achieves superior performance on benchmarks like AIME 2025 and HMMT, rivalling models such as Google’s Gemini 2.5 Fl..."
2,Who developed Qwen?,before,3,Qwen -> developed by -> Alibaba,0.957,True,"Alibaba Group Holding (BABA Quick QuoteBABA - Free Report) too introduced the QwQ-32B model, an AI system that rivals DeepSeek but requires only a fraction of the data."
3,Who developed Qwen?,after,1,Qwen -> developed by -> Alibaba,0.998,True,"Part of the underlying AI for GeoGPT is Qwen, a large language model built by the Chinese tech company Alibaba."
4,Who developed Qwen?,after,2,Qwen -> developed by -> Alibaba,0.998,True,"Positive Sentiment: Alibaba’s ecosystem is showing continued progress in AI and digital infrastructure, including reports that its Qwen model beat OpenAI and Google r..."
...,...,...,...,...,...,...,...
67,Under which license is Gemma released?,before,2,Gemma -> licensed under -> Apache 2.0,0.020,False,"Hot on Gemma 3’s heels: Shortly after Gemma 3 became available, Mistral released Small 3.1 (24 billion parameters), a vision-language model with open weights, under a..."
68,Under which license is Gemma released?,before,3,Gemma -> licensed under -> Apache 2.0,0.992,True,Hugging Face | GitHub | Launch Blog | Documentation License: Apache 2.0 | Authors: Google DeepMind Gemma is a family of open models built by Google DeepMind.
69,Under which license is Gemma released?,after,1,Gemma -> licensed under -> Apache 2.0,0.995,True,"(Anthropic) Gemma 4 open models, now with Apache 2.0 licenses Google released Gemma 4, a family of open-weights models in four sizes: Effective 2B, Effective 4B, 26B ..."
70,Under which license is Gemma released?,after,2,Gemma -> licensed under -> Apache 2.0,0.992,True,Hugging Face | GitHub | Launch Blog | Documentation License: Apache 2.0 | Authors: Google DeepMind Gemma is a family of open models built by Google DeepMind.


**Reading.** "Supported" in the summary is measured with the same check that filters the passages, so the share
after the revision is 1 by construction. The informative numbers are the passages *before* that the check rejects,
the passages that changed, and whether every question still finds evidence. Rejected passages include the cases
named in the feedback, e.g. *DeepSeek -> headquartered in -> United States* from *"... companies, largely
headquartered in the United States"*, *Gemma -> licensed under -> Apache 2.0* from a sentence about Mistral's
license, and *DeepSeek -- competes with -- Google* from a sentence comparing groups of companies; but also clear
rivalry sentences with four names (Section 3.3). The two competition questions return no graph evidence after the
revision, by design (Section 3.6): they go to text retrieval. Lexicon facts now appear as background, separate from
the cited passages. An independent check
needs people: the cell below writes a blind review sheet with the passages of both systems in random order and
without the system name. Each team member labels whether the passage states the relation, and the labels are read
back in Section 4.1 once they exist.

In [21]:
REVIEW_SHEET = ARTIFACT_DIR / "retrieval_support_review.xlsx"
REVIEW_KEY = ARTIFACT_DIR / "retrieval_support_review_key.csv"
sheet = (retrieval_check.drop_duplicates(["question", "relation_id"])
         .sample(frac=1.0, random_state=SEED).reset_index(drop=True))
sheet.insert(0, "item_id", [f"q{i:03d}" for i in range(1, len(sheet) + 1)])
if not REVIEW_SHEET.exists():   # never overwrite a sheet that may already contain labels
    review_columns_out = ["item_id", "question", "relation", "passage", "published_at", "document_id"]
    sheet[review_columns_out].assign(passage_states_relation="", reviewer="", notes="").to_excel(
        REVIEW_SHEET, sheet_name="Review", index=False)
    membership = retrieval_check.groupby(["question", "relation_id"])["system"].agg(lambda s: "+".join(sorted(set(s))))
    sheet[["item_id", "question", "relation_id"]].join(membership, on=["question", "relation_id"]).to_csv(REVIEW_KEY, index=False)
    print(f"Review sheet written: {REVIEW_SHEET.relative_to(ROOT)} ({len(sheet)} passages)")
else:
    print(f"Review sheet exists and is kept: {REVIEW_SHEET.relative_to(ROOT)}")

Review sheet exists and is kept: artifacts/stage3_wp2/kg_revision/retrieval_support_review.xlsx


### 4.1 Human check of the retrieved passages

Each labelled copy of the sheet is saved as `retrieval_support_review_<reviewer>.xlsx` (`passage_states_relation` =
`TRUE`, `FALSE` or `AMBIGUOUS`). Labels are matched to the passages by question, relation and passage text, because
item IDs change when the sheet is generated again. The reviewer saw neither the system nor the support score.
`AMBIGUOUS` labels are reported but not counted in the share.

In [22]:
LABEL_FILES = sorted(ARTIFACT_DIR.glob("retrieval_support_review_*.xlsx"))
MATCH = ["question", "relation", "passage"]
human = pd.concat([pd.read_excel(f, sheet_name="Review") for f in LABEL_FILES], ignore_index=True) if LABEL_FILES else pd.DataFrame()
if human.empty:
    print("No labels yet: the human check of the retrieved passages is pending.")
else:
    human["human_label"] = human["passage_states_relation"].astype(str).str.strip().str.upper()
    human = human[human["human_label"].isin(["TRUE", "FALSE", "AMBIGUOUS"])].drop_duplicates(MATCH + ["reviewer"])
    checked = retrieval_check.merge(human[MATCH + ["human_label", "reviewer", "notes"]], on=MATCH, how="left")
    print(f"Label files: {[f.name for f in LABEL_FILES]}; reviewers: {sorted(human['reviewer'].dropna().unique())}; "
          f"retrieved passages without a label: {int(checked['human_label'].isna().sum())}")

    rows = []
    for system in ["before", "after"]:
        part = checked[checked["system"].eq(system)]
        counts = part["human_label"].value_counts()
        true, false = int(counts.get("TRUE", 0)), int(counts.get("FALSE", 0))
        low, high = wilson(true, true + false)
        rows.append({"system": system, "passages": len(part), "TRUE": true, "FALSE": false,
                     "AMBIGUOUS": int(counts.get("AMBIGUOUS", 0)),
                     "share stating the relation": round(true / (true + false), 3), "95% CI": f"{low:.2f}-{high:.2f}"})
    human_summary = pd.DataFrame(rows)
    display(human_summary)

    # Does the support check agree with the reviewer? (unique passages)
    unique_passages = checked.drop_duplicates(["question", "relation_id"])
    display(pd.crosstab(unique_passages["supported"].map({True: "check: supported", False: "check: not supported"}),
                        unique_passages["human_label"], margins=True).rename_axis(index=None, columns="human label"))
    with pd.option_context("display.max_colwidth", 150):
        display(unique_passages[unique_passages["human_label"].isin(["FALSE", "AMBIGUOUS"])]
                [["system", "relation", "supported", "human_label", "notes", "passage"]].sort_values(["human_label", "system"]))

Label files: ['retrieval_support_review_jihyeon.xlsx']; reviewers: ['Jihyeon']; retrieved passages without a label: 0


,system,passages,TRUE,FALSE,AMBIGUOUS,share stating the relation,95% CI
0,before,39,32,5,2,0.865,0.72-0.94
1,after,33,27,1,5,0.964,0.82-0.99


human label,AMBIGUOUS,FALSE,TRUE,All
check: not supported,0,3,7,10
check: supported,6,3,46,55
All,6,6,53,65


,system,relation,supported,human_label,notes,passage
12,after,GB200 NVL72 -> manufactured by -> NVIDIA,True,AMBIGUOUS,"GB200 NVL72 is presented in an NVIDIA context, but the passage does not explicitly identify its manufacturer.","At NVIDIA GTC Washington, D.C., NVIDIA founder and CEO Jensen Huang highlighted how GB200 NVL72 delivers 10x the performance of NVIDIA Hopper for ..."
13,after,A100 -> manufactured by -> NVIDIA,True,AMBIGUOUS,NVIDIA is described as providing A100 GPUs; manufacturing is inferred rather than stated.,"In the data center segment, NVIDIA provides high-throughput accelerators—such as the NVIDIA A100 and H100 Tensor Core GPUs—and integrated systems ..."
14,after,GeForce RTX -> manufactured by -> NVIDIA,True,AMBIGUOUS,"NVIDIA is expanding GeForce RTX gaming to more devices, but the passage does not explicitly establish manufacturing.","In addition, NVIDIA is bringing GeForce RTX gaming to more devices with new GeForce NOW Apps for Linux PC and Amazon Fire TV."
71,after,Gemma -> licensed under -> Apache 2.0,True,AMBIGUOUS,"License and Gemma appear together in compressed metadata, but the specific license scope is unclear.",Hugging Face | GitHub | Launch Blog | Documentation License: Apache 2.0 | Authors: Google DeepMind Gemma is a family of open models built by Googl...
30,before,Kimi -> outperforms -> Claude,True,AMBIGUOUS,The passage reports a claim that Kimi K2 outperforms Claude on certain metrics; the claim is not independently established by this passage.,"Separately, Alibaba-backed startup Moonshot released its new Kimi K2 AI model late on Friday, which claims to outperform OpenAI's ChatGPT and Anth..."
68,before,Gemma -> licensed under -> Apache 2.0,True,AMBIGUOUS,"License and Gemma appear together in compressed metadata, but the specific license scope is unclear.",Hugging Face | GitHub | Launch Blog | Documentation License: Apache 2.0 | Authors: Google DeepMind Gemma is a family of open models built by Googl...
20,after,Gemma -> developed by -> Google,True,FALSE,Passage names DolphinGemma rather than the Gemma family generally.,"DolphinGemma, a large language model developed by Google, is helping scientists study how dolphins communicate — and hopefully find out what they'..."
2,before,Qwen -> developed by -> Alibaba,True,FALSE,"Passage identifies Alibaba’s QwQ-32B, not Qwen directly.","Alibaba Group Holding (BABA Quick QuoteBABA - Free Report) too introduced the QwQ-32B model, an AI system that rivals DeepSeek but requires only a..."
38,before,DeepSeek -> headquartered in -> united_states,False,FALSE,"The United States headquarters description refers to established AI companies compared with DeepSeek, not to DeepSeek itself.","In early 2025, the Chinese AI company DeepSeek made international news upon releasing a large language model (LLM) that appeared to outperform the..."
54,before,Nemotron -> available on -> GitHub,True,FALSE,"GitHub availability applies to NeMo Switchyard, not Nemotron.","NeMo Switchyard is available on GitHub. Visit the Nemotron 3.5 Lightning, NeMo Switchyard and Jetson AI technical blogs to get started."


In [23]:
if not human.empty:
    rejected_accepted = unique_passages[~unique_passages["supported"].astype(bool) & unique_passages["human_label"].eq("TRUE")]
    accepted_rejected = unique_passages[unique_passages["supported"].astype(bool) & unique_passages["human_label"].eq("FALSE")]
    after_ambiguous = checked[checked["system"].eq("after") & checked["human_label"].eq("AMBIGUOUS")]
    before_h, after_h = human_summary.set_index("system").loc["before"], human_summary.set_index("system").loc["after"]
    display(Markdown(f'''
**Reading.** In the blind check, {before_h["share stating the relation"]:.0%}
of the passages returned before and {after_h["share stating the relation"]:.0%} of those returned after the revision
state their relation ({before_h["95% CI"]} and {after_h["95% CI"]}); for these {len(QUESTIONS)} questions most
passages were already correct. Where the check and the reviewer differ, the cases show what the check does and does
not catch:
- the check rejected {len(rejected_accepted)} passages the reviewer accepted, {int(rejected_accepted["relation"].str.contains("competes with").sum())}
  of them `competes with` sentences removed by the list rule (e.g. *"DeepSeek said ... its R1 model competes with
  OpenAI's o1"*). The list rule is too strict for sentences that state a rivalry between two named players, one
  more reason not to answer competition questions from the graph (Section 3.6);
- the check accepted {len(accepted_rejected)} passages the reviewer rejected because the sentence is about a single
  product or another entity, not the node: *DolphinGemma* for Gemma, *QwQ-32B* for Qwen, *NeMo Switchyard* for
  Nemotron. This is the family-level linking error of Section 3.4, which the sentence check cannot see;
- {int(after_ambiguous["relation"].str.contains("manufactured by").sum())} of the {len(after_ambiguous)} `AMBIGUOUS`
  passages after the revision are `manufactured by` (*"NVIDIA provides ... A100 and H100 Tensor Core GPUs"*): the
  NLI model accepts a relation the sentence implies, while the reviewer asked for one it states. Implied relations
  are kept, and the answer shows the sentence, so the wording stays visible.
'''))


**Reading.** In the blind check, 86%
of the passages returned before and 96% of those returned after the revision
state their relation (0.72-0.94 and 0.82-0.99); for these 13 questions most
passages were already correct. Where the check and the reviewer differ, the cases show what the check does and does
not catch:
- the check rejected 7 passages the reviewer accepted, 6
  of them `competes with` sentences removed by the list rule (e.g. *"DeepSeek said ... its R1 model competes with
  OpenAI's o1"*). The list rule is too strict for sentences that state a rivalry between two named players, one
  more reason not to answer competition questions from the graph (Section 3.6);
- the check accepted 3 passages the reviewer rejected because the sentence is about a single
  product or another entity, not the node: *DolphinGemma* for Gemma, *QwQ-32B* for Qwen, *NeMo Switchyard* for
  Nemotron. This is the family-level linking error of Section 3.4, which the sentence check cannot see;
- 3 of the 5 `AMBIGUOUS`
  passages after the revision are `manufactured by` (*"NVIDIA provides ... A100 and H100 Tensor Core GPUs"*): the
  NLI model accepts a relation the sentence implies, while the reviewer asked for one it states. Implied relations
  are kept, and the answer shows the sentence, so the wording stays visible.


## 5. Knowledge Graph v2: Outputs for Graph-Enhanced RAG

`outputs/stage3_kg_v2/` contains everything the WP2 retriever needs; `GraphEvidenceRetriever.from_directories(dir, dir)`
loads it. Small tables and the data dictionary are also written to `artifacts/stage3_wp2/kg_revision/`.

In [24]:
edge_columns = [c for c in collapsed.columns] + ["relation_reliability",
    "knowledge_layer", "has_article_claim", "has_name_pattern", "in_lexicon", "lexicon_agreement", "article_sentences",
    "article_documents", "article_first_seen", "article_last_seen", "support_status", "support_max", "support_mean",
    "scored_sentences", "supported_sentences", "support_disagrees_with_review", "retrieval_eligible"]
edges_v2 = edges[edge_columns]
assertion_columns = [c for c in assertions_v2.columns if c not in ("key",)]
tables = {
    "nodes": nodes,
    "assertions": assertions_v2[assertion_columns],
    "retrieval_assertions": retrieval_assertions[assertion_columns],
    "edges": edges_v2,
    "background_facts": background,
    "text_units": text_units,
    "documents": documents,
    "lexicon_conflicts": conflict_support.drop(columns=["key"]),
}
for name, table in tables.items():
    table.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
# typed_edges_assertions.parquet: the file name GraphEvidenceRetriever.from_directories reads
retrieval_assertions[assertion_columns].to_parquet(OUTPUT_DIR / "typed_edges_assertions.parquet", index=False)

graph = nx.MultiDiGraph(graph_view="typed_v2", schema_version="stage3-wp2-open-models-kg-v2.0")
for row in edges_v2.itertuples(index=False):
    for entity_id in (row.subject_id, row.object_id):
        graph.add_node(entity_id, label=str(labels_of.get(entity_id, entity_id)))
    graph.add_edge(row.subject_id, row.object_id, key=row.predicate, knowledge_layer=row.knowledge_layer,
                   support_status=str(row.support_status), retrieval_eligible=bool(row.retrieval_eligible),
                   article_documents=int(row.article_documents), predicate_group=str(row.predicate_group))
nx.write_graphml(graph, OUTPUT_DIR / "typed_graph_v2.graphml")

evaluation_tables = {"edges_v2": edges_v2.drop(columns=["relation_ids", "document_ids", "text_unit_ids", "domains"]),
                     "agreement_with_labels": agreement, "agreement_by_predicate": by_predicate.reset_index(),
                     "label_disagreements": disagreements, "retrieval_check": retrieval_check,
                     "centrality_with_without_lexicon": rank_compare.reset_index(),
                     "lexicon_conflicts": conflict_support.drop(columns=["key"])}
for name, table in evaluation_tables.items():
    table.to_csv(ARTIFACT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

['KG_V2_DATA_DICTIONARY.md', 'assertions.parquet', 'background_facts.parquet', 'documents.parquet', 'edges.parquet', 'lexicon_conflicts.parquet', 'manifest.json', 'nodes.parquet', 'retrieval_assertions.parquet', 'text_units.parquet', 'typed_edges_assertions.parquet', 'typed_graph_v2.graphml']


In [25]:
DATA_DICTIONARY = f'''# Knowledge graph v2 (Stage 3 WP2): data dictionary

Built from the submitted Stage 1 knowledge graph by `notebooks/stage3_wp2_kg_revision.ipynb`. Node IDs, relation IDs
and text unit IDs are those of Stage 1. All dates are article publication dates.

## `edges` - one row per triple (all Stage 1 graph edges)
Stage 1 columns, plus:
| column | description |
|---|---|
| `knowledge_layer` | `article_claim` (stated in an article sentence), `name_pattern` (model name with size -> family), `curated_lexicon` (WP1 lexicon only, no sentence) |
| `has_article_claim`, `has_name_pattern`, `in_lexicon` | which sources contain the triple |
| `lexicon_agreement` | `lexicon_and_articles` or `lexicon_only` for lexicon triples |
| `article_sentences`, `article_documents`, `article_first_seen`, `article_last_seen` | support counted on article sentences only |
| `support_status` | `supported` (at least one sentence entails the relation), `not_supported`, `no_passage` (lexicon only) |
| `support_max`, `support_mean` | best and mean NLI support score of the sentences |
| `scored_sentences`, `supported_sentences` | sentences checked / supporting |
| `support_disagrees_with_review` | confirmed by a person but no supporting sentence (kept) |
| `relation_reliability` | `low` for `competes with` (cannot be validated at sentence level, Section 3.6), else `standard` |
| `retrieval_eligible` | used by the retriever: article claim or name pattern, supported or confirmed by a person, and `relation_reliability = standard` |

## `assertions` - one row per evidence sentence (`retrieval_assertions`: the rows of eligible edges)
Stage 1 columns (`relation_id`, `subject_id`, `predicate`, `object_id`, `document_id`, `text_unit_id`, `evidence_text`,
`published_at`, `extraction_method`, `rule_score`), plus `knowledge_source` (layer of the row), `support_score`
(NLI entailment probability, {NLI_MODEL}), `contradiction_score`, `n_actors` (named models / organisations in the
sentence), `list_sentence` (n_actors >= 3), `supported` (support_score >= {SUPPORT_THRESHOLD}, and for
`competes with` not a list sentence), `hypothesis` (the sentence the relation was checked as), `retrieval_eligible`.

## `background_facts`
Lexicon facts (`subject_id`, `predicate`, `object_id`, `knowledge_source = curated_lexicon`). Returned by the
retriever as `background`, never as cited evidence.

## `text_units`, `documents`
Sentences (`text_unit_id` = `doc_id:sentence_id`) and articles (publication date, title, URL, domain).

## `lexicon_conflicts`
Article relations that Stage 1 excluded because they name another developer of a lexicon family, with their
support status. They stay excluded.

## `typed_graph_v2.graphml`
All edges with `knowledge_layer`, `support_status`, `retrieval_eligible`.
'''
for directory in [OUTPUT_DIR, ARTIFACT_DIR]:
    (directory / "KG_V2_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")

file_hash = lambda path: sha256(Path(path).read_bytes()).hexdigest()
manifest = {
    "schema_version": "stage3-wp2-open-models-kg-v2.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "nli_model": NLI_MODEL, "support_threshold": SUPPORT_THRESHOLD, "list_min_actors": 3,
    "not_retrieval_eligible_predicates": LOW_RELIABILITY,
    "edges": int(len(edges_v2)), "retrieval_eligible_edges": int(edges_v2["retrieval_eligible"].sum()),
    "layers": edges_v2["knowledge_layer"].value_counts().to_dict(),
    "support_status": edges_v2["support_status"].value_counts().to_dict(),
    "inputs": {str(p.relative_to(STAGE1 if p.is_relative_to(STAGE1) else ROOT)): file_hash(p) for p in
               [KG_DIR / "nodes.parquet", KG_DIR / "typed_edges_assertions.parquet", KG_DIR / "typed_edges_collapsed.parquet",
                KG_DIR / "correction" / "relations_all_tiers.csv", KG_DIR / "correction" / "evidence_status.csv",
                REVIEW_DIR / "relation_review_corrected_seed2026.xlsx", REVIEW_DIR / "insight_relation_check.csv"]},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "networkx": nx.__version__},
}
for directory in [OUTPUT_DIR, ARTIFACT_DIR]:
    (directory / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

reloaded = GraphEvidenceRetriever.from_directories(OUTPUT_DIR, OUTPUT_DIR)
assert reloaded.retrieve("Who developed Qwen?", predicates=["developed by"], min_support=SUPPORT_THRESHOLD)["evidence"]
print(json.dumps({k: v for k, v in manifest.items() if k != "inputs"}, indent=2))

{
  "schema_version": "stage3-wp2-open-models-kg-v2.0",
  "created_at_utc": "2026-10-24T21:47:50+00:00",
  "nli_model": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
  "support_threshold": 0.5,
  "list_min_actors": 3,
  "not_retrieval_eligible_predicates": {
    "competes with": "cannot be validated at sentence level: lists of rivals, 6 of 22 sampled relations correct"
  },
  "edges": 999,
  "retrieval_eligible_edges": 805,
  "layers": {
    "article_claim": 846,
    "name_pattern": 135,
    "curated_lexicon": 18
  },
  "support_status": {
    "supported": 710,
    "not_supported": 136,
    "name_pattern": 135,
    "no_passage": 18
  },
  "versions": {
    "python": "3.12.13",
    "pandas": "3.1.0",
    "networkx": "3.7"
  }
}


## 6. Summary and Remaining Limitations

In [26]:
random_all = decided[decided["label_set"].eq("random sample")]
kept_all = random_all[random_all["kept"]]
low_before, high_before = wilson(int(random_all["is_true"].sum()), len(random_all))
low_after, high_after = wilson(int(kept_all["is_true"].sum()), len(kept_all))
before_summary, after_summary = summary.loc["before"], summary.loc["after"]
n_mixed = int(edges["lexicon_agreement"].eq("lexicon_and_articles").sum())
n_lexicon_only = int(edges["lexicon_agreement"].eq("lexicon_only").sum())
rejected_before = int(before_summary["passages"] - before_summary["supported"])
human_text = ("a blind human check of the passages is pending" if human.empty else
              f"in a blind human check, {human_summary.loc[0, 'share stating the relation']:.0%} of the passages before and "
              f"{human_summary.loc[1, 'share stating the relation']:.0%} after state their relation (Section 4.1)")
display(Markdown(f'''
**What changed for Stage 3**

| feedback | knowledge graph v2 |
|---|---|
| keep lexicon knowledge separate | every edge has a `knowledge_layer`; {n_mixed} edges that Stage 1 labelled Tier A are article claims; {n_lexicon_only} lexicon-only facts are returned as `background` and never cited; graph metrics are computed on validated article claims only |
| validate the relations used for retrieval | `competes with`, which cannot be validated, is not used for retrieval or graph metrics; every other article sentence was checked against its relation; on the human random sample, precision rises from {random_all["is_true"].mean():.2f} ({low_before:.2f}-{high_before:.2f}) to {kept_all["is_true"].mean():.2f} ({low_after:.2f}-{high_after:.2f}) while {kept_all["is_true"].sum() / random_all["is_true"].sum():.0%} of the correct relations are kept; {int(edges["retrieval_eligible"].sum())} of {len(edges)} edges are used for retrieval |
| passages must support the relation | the retriever cites only sentences that state the relation, ranked by support score; in the repeated retrieval check {rejected_before} of {int(before_summary["passages"])} passages returned before do not pass the check; {int(after_summary["questions_answered"])} of {len(QUESTIONS)} questions return graph evidence, the competition questions go to text retrieval; {human_text} |

**Remaining limitations**
- **The check is a filter, not a proof.** About a quarter of the kept relations in the random sample are still
  wrong, mostly fragment nodes and possessive lists that the NLI model accepts (Section 3.4). Retrieval therefore
  shows the passage with every relation, so that the answer can cite the sentence rather than the edge.
- **No graph answer for competition.** `competes with` stays in the graph with low reliability but answers no
  question; competition questions depend on text retrieval, and the graph cannot show how rivalry is reported
  over time.
- **Entity linking is not repaired here.** The check removes relations whose sentence does not state them, but
  family-level linking (*Llama Nemotron* -> Llama) still needs the lexicon filter, and fragment nodes stay in the
  graph. A mention-level linking step would address both at the source.
'''))


**What changed for Stage 3**

| feedback | knowledge graph v2 |
|---|---|
| keep lexicon knowledge separate | every edge has a `knowledge_layer`; 15 edges that Stage 1 labelled Tier A are article claims; 18 lexicon-only facts are returned as `background` and never cited; graph metrics are computed on validated article claims only |
| validate the relations used for retrieval | `competes with`, which cannot be validated, is not used for retrieval or graph metrics; every other article sentence was checked against its relation; on the human random sample, precision rises from 0.60 (0.50-0.70) to 0.76 (0.65-0.85) while 95% of the correct relations are kept; 805 of 999 edges are used for retrieval |
| passages must support the relation | the retriever cites only sentences that state the relation, ranked by support score; in the repeated retrieval check 10 of 39 passages returned before do not pass the check; 11 of 13 questions return graph evidence, the competition questions go to text retrieval; in a blind human check, 86% of the passages before and 96% after state their relation (Section 4.1) |

**Remaining limitations**
- **The check is a filter, not a proof.** About a quarter of the kept relations in the random sample are still
  wrong, mostly fragment nodes and possessive lists that the NLI model accepts (Section 3.4). Retrieval therefore
  shows the passage with every relation, so that the answer can cite the sentence rather than the edge.
- **No graph answer for competition.** `competes with` stays in the graph with low reliability but answers no
  question; competition questions depend on text retrieval, and the graph cannot show how rivalry is reported
  over time.
- **Entity linking is not repaired here.** The check removes relations whose sentence does not state them, but
  family-level linking (*Llama Nemotron* -> Llama) still needs the lexicon filter, and fragment nodes stay in the
  graph. A mention-level linking step would address both at the source.
